In [ ]:
# One-time: the project was renamed. If an older Drive folder exists, rename it so cached data and weights are kept.
import os
try:
    from google.colab import drive; drive.mount('/content/drive')
    _old, _new = '/content/drive/MyDrive/' + 'exo' + 'lens', '/content/drive/MyDrive/anantara'
    if os.path.isdir(_old) and not os.path.exists(_new): os.rename(_old, _new); print('Renamed old Drive folder ->', _new)
except Exception: pass

# Model A (real data) – 2D CNN trained on real JWST observations only
Connects **astroquery.mast → real JWST spectra → 2D images → CNN**.
You supply the answers (which molecules each planet is known to have) from published papers; the code fetches the data and trains.

**Honest expectation:** with ~5–15 planets this is a *proof of concept*. The leave-one-planet-out test below tells you how well it really generalises — expect weak results, and report them as such.

In [ ]:
!pip -q install astroquery astropy scipy
import os, time, glob, numpy as np, pandas as pd, matplotlib.pyplot as plt
import torch, torch.nn as nn, torch.nn.functional as F
from astropy.io import fits
from astroquery.mast import Observations
from google.colab import drive; drive.mount('/content/drive')
DATA='/content/drive/MyDrive/anantara/mast_jwst'; os.makedirs(DATA,exist_ok=True)
dev='cuda' if torch.cuda.is_available() else 'cpu'
MOLS=['H2O','CO2','CH4','SO2']

## 1. Your planets + answers
Use **one instrument mode** so every image covers the same wavelengths. Below: NIRSpec **G395H** (≈2.9–5.1 µm: CH₄ 3.3, SO₂ 4.0, CO₂ 4.3).
Each row = (MAST target name, H2O, CO2, CH4, SO2) with 1 = detected in the published paper, 0 = not detected.
**Verify every row against its paper, and add more planets.** (WASP-39 is a famous G395H target; other candidates to look up: WASP-107, WASP-80, HD 209458, HAT-P-18 – check which have G395H time-series data on MAST.)

In [ ]:
INSTRUMENT_FILTER = 'G395H'
WAVE_LO, WAVE_HI  = 2.9, 5.1
LABELS = [
    # target,   H2O, CO2, CH4, SO2
    ('WASP-39',  1,   1,   0,   1),     # <- verify against the JWST ERS papers
    # ('WASP-xx', 0, 0, 0, 0),           # add more rows here
]

## 2. Download from MAST (`astroquery.mast`) and stack the spectra into a wavelength × time table

In [ ]:
def download_target(target):
    obs=Observations.query_criteria(obs_collection='JWST',target_name=target,intentType='SCIENCE')
    obs=obs[[INSTRUMENT_FILTER in str(f) for f in obs['filters']]]
    if len(obs)==0: print(target,': no',INSTRUMENT_FILTER,'observations found'); return []
    prods=Observations.get_product_list(obs)
    prods=prods[[str(f).endswith('_x1dints.fits') for f in prods['productFilename']]]   # one extracted spectrum per integration
    if len(prods)==0: print(target,': no x1dints products (needs manual reduction)'); return []
    print(target,':',len(prods),'files, first obs',obs['obs_id'][0])
    m=Observations.download_products(prods,download_dir=DATA)
    return [p for p in m['Local Path']]

def load_matrix(files):
    # group by detector (nrs1 = short wavelengths, nrs2 = long), stack segments in time, then join detectors in wavelength
    parts=[]
    for det in ('nrs1','nrs2'):
        fs=sorted(f for f in files if det in os.path.basename(f))
        if not fs: continue
        W=None; cols=[]
        for f in fs:
            with fits.open(f) as h:
                sp=[x for x in h if x.name=='EXTRACT1D']
                W=sp[0].data['WAVELENGTH']; cols.append(np.stack([s.data['FLUX'] for s in sp],axis=1))
        parts.append((W,np.concatenate(cols,axis=1)))
    n=min(p[1].shape[1] for p in parts)                      # equal time length across detectors
    return np.concatenate([p[0] for p in parts]), np.concatenate([p[1][:,:n] for p in parts],axis=0)

In [ ]:
import numpy as np
from scipy.ndimage import zoom
NW, NT = 64, 64
def to_image(wave, flux, wave_lo, wave_hi):
    """wave (n_wave,), flux (n_wave, n_time) -> (64,64) float32 image, or None if unusable."""
    order = np.argsort(wave); wave, flux = wave[order], flux[order]
    keep = (wave >= wave_lo) & (wave <= wave_hi) & (np.isfinite(flux).mean(axis=1) > 0.8)
    wave, flux = wave[keep], flux[keep]
    if flux.shape[0] < 16 or flux.shape[1] < 32: return None
    row_med = np.nanmedian(flux, axis=1, keepdims=True)
    flux = np.where(np.isfinite(flux), flux, row_med)                   # fill NaNs
    n = flux.shape[1]
    oot = np.r_[0:int(.2*n), int(.8*n):n]                               # assumed out-of-transit columns
    rel = flux / np.median(flux[:, oot], axis=1, keepdims=True) - 1
    rel = np.clip(rel, np.percentile(rel, 0.5), np.percentile(rel, 99.5))   # tame outliers
    img = zoom(rel, (NW/rel.shape[0], NT/rel.shape[1]), order=1)
    depth = -img[:, 24:40].mean()
    if not depth > 0: return None                                       # no dip in the centre -> bad assumption
    return (img / depth).astype(np.float32)

def augment(img, rng):
    x = np.roll(img, rng.integers(-3, 4), axis=1)                       # small time shift
    x = np.roll(x, rng.integers(-1, 2), axis=0)                         # tiny wavelength shift
    x = x * rng.uniform(0.9, 1.1)                                       # gain jitter
    x = x + rng.normal(0, rng.uniform(0.02, 0.2), x.shape)              # extra noise
    return x.astype(np.float32)

In [ ]:
images={}; labels={}
for target,*lab in LABELS:
    files=download_target(target)
    if not files: continue
    try:
        wave,flux=load_matrix(files); img=to_image(wave,flux,WAVE_LO,WAVE_HI)
    except Exception as e: print(target,'failed:',e); continue
    if img is None: print(target,': image unusable (check transit is roughly centred)'); continue
    images[target]=img; labels[target]=np.array(lab,dtype=np.float32)
print('usable planets:',list(images))
fig,ax=plt.subplots(1,max(len(images),1),figsize=(4*max(len(images),1),3.5),squeeze=False)
for a,(k,im) in zip(ax[0],images.items()):
    a.imshow(im,aspect='auto',extent=[-1,1,WAVE_HI,WAVE_LO]); a.set_title(k); a.set_ylabel('µm')
plt.show()   # LOOK at these: you should see a dark band (the dip) in the middle columns

## 3. The CNN (same 2D architecture as the simulated Model A)
Set `PRETRAINED` to `'model_a_2d.pt'` only if you want to start from the simulated-data weights; `None` = real data only (what you asked for).

In [ ]:
PRETRAINED=None
class SpecCNN2D(nn.Module):
    def __init__(s):
        super().__init__()
        blk=lambda i,o: nn.Sequential(nn.Conv2d(i,o,3,padding=1),nn.BatchNorm2d(o),nn.ReLU(),nn.MaxPool2d(2))
        s.features=nn.Sequential(blk(1,8),blk(8,16),blk(16,32),nn.Conv2d(32,32,3,padding=1),nn.ReLU())
        s.head=nn.Sequential(nn.Dropout(.5),nn.Linear(32,4))
    def forward(s,x): return s.head(s.features(x).mean((-2,-1)))

def train_model(names,epochs=40,per_epoch=128,seed=0):
    rng=np.random.default_rng(seed); torch.manual_seed(seed)
    m=SpecCNN2D().to(dev)
    if PRETRAINED and os.path.exists(PRETRAINED): m.load_state_dict(torch.load(PRETRAINED),strict=False)
    opt=torch.optim.AdamW(m.parameters(),1e-3,weight_decay=5e-2); lossf=nn.BCEWithLogitsLoss()
    for ep in range(epochs):
        m.train()
        for _ in range(per_epoch//16):
            pick=rng.choice(names,16)                                   # every planet gets sampled, new random augmentation each time
            xb=torch.tensor(np.stack([augment(images[n],rng) for n in pick]))[:,None].to(dev)
            yb=torch.tensor(np.stack([labels[n] for n in pick])).to(dev)
            opt.zero_grad(); lossf(m(xb),yb).backward(); opt.step()
    return m

def predict(m,img):
    m.eval()
    with torch.no_grad(): return torch.sigmoid(m(torch.tensor(img)[None,None].to(dev)))[0].cpu().numpy()

## 4. Honest test: leave-one-planet-out
Train on all planets except one, then predict the planet it never saw. Repeat for each planet. (Never test on a planet you trained on – the model would just memorise it.)

In [ ]:
names=list(images); rows=[]
if len(names)<3: print('Need at least 3 usable planets for this test – add more rows to LABELS.')
else:
    for held in names:
        m=train_model([n for n in names if n!=held])
        p=predict(m,images[held]); rows.append([held]+list(np.round(p,2))+['|']+list(labels[held].astype(int)))
    print(pd.DataFrame(rows,columns=['planet']+[f'pred_{x}' for x in MOLS]+['|']+[f'true_{x}' for x in MOLS]))
    P=np.array([r[1:5] for r in rows]); Y=np.array([labels[r[0]] for r in rows])
    print('overall accuracy (threshold 0.5):',((P>.5)==Y).mean().round(2),' – compare with always-guessing-the-majority:',max(Y.mean(),1-Y.mean()).round(2))

## 5. Final model on all planets, and use it on a new one

In [ ]:
if images:
    final=train_model(list(images),epochs=60); torch.save(final.state_dict(),f'{DATA}/model_a_real.pt')
    # new planet: files=download_target('NAME'); wave,flux=load_matrix(files); img=to_image(wave,flux,WAVE_LO,WAVE_HI)
    # print(dict(zip(MOLS,predict(final,img).round(2))))